<a href="https://colab.research.google.com/github/GabrieleCirillo/desktop-tutorial/blob/main/Copia_di_Hackathon_NameGen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

For this task, you are going to train a name generator! Ever wondered where **Gwyneth** Paltrow got her name, or how Hawaiian woman Janice **Keihanaikukauakahihuliheekahaunaele** came to be? Probably from this very generator.

Let's get going!

### Overall pipeline

- **Training**

  You must implement a model that, given the beginning of a word, predicts the next letter.

  For example, if the model `M` is trained on the word `"house"`, the trained model should output `M("h") = "o"`, `M("o") = "u"`, and so on.

  Of course, the model will be trained on an entire dataset of words. Therefore, the output of `M("h")` should be *not* deterministic, meaning that ***each time you run the trained model, you should get a different result***.

- **Inference**

  At test time, the model generates only one new letter, given an input letter.

  However, you can still generate entire words by an approach that is as simple as it's weird-sounding: *ancestral sampling*:

  1. Select an initial letter, say `"b"`.
  2. Predict the next letter, say `M("b") = "r"`.
  3. Concatenate: `"br"`.
  4. Iterate: go back to step 2, and predict `M("r") = "e"`.
  5. Concatenate: `"bre"`.
  6. Keep going, you might eventually get the word `"bread"`.

- **Delimiters**

  Of course, ancestral sampling will keep generating *ad infinitum*.

  To avoid this, you also want to train the model to *end* the generation.

  This is easily done: simply take your training data and augment it with beginning / end delimiters:

  `"house" -> ".house."`

  We chose `"."` here, but you can choose your own.

  This way, you can ask the model to create a new word completely from scratch, by simply starting inference with `M(".") = ...`. And you can stop generating whenever you get `M(...) = "."`.

### What you'll need

- A list of names: your training data (*see names.txt*)
- A way of encoding names to numbers, and viceversa
- A model that generates new names (duh)

# The power of Positions

Before Training a more complex model, which we choose to be a sort of CBOW mixed with a VAE, we tried to crack the problem with a simpler model.
We tried to infer the probabilities of a character given the previous one only with counting the number of occurence of a particular $x_{t+1}$ given $x_t$
and then using Softmax (the boltzman-gibbs probability)
...

The model wasn't really good, but it seems to be improved with the use a simple trick:
We encoded the positions of the single characters directly into the strings, and then tokenized the new charcaters:
### Example
$"ciao"$ -> $"c.1 \ i.2 \ a.3 \ o.4 §" $

The character $"§"$ reppresent the end of a name, so that the model can indeed put an end to the name creation.
This method seems to have really improved the names generation.
We know that we could also use a positional encoding instead, but choose the simplest way.

### A Consideration
-*This idea, I found was particularly good with music, for example to generate chord progressions starting from strings like C.1 C.2 Dm.3 G.4 C.1 G.2 F.3 ... the models easily learn the importance of that G chord in the 3th position threating it as a totally different token from G.2*


In [14]:
import numpy as np
import torch
import math
import random as rd

def softmaxdict(X: dict, beta=1.0):
    maxxed = []
    values = X.values()
    Z = np.exp( (np.array(list(values))  - max(values)) * beta ).sum()
    for couple in X.items():
        # maxxed.append((couple[0], np.exp(couple[1] - max(values)) / Z))
        maxxed.append((couple[0], np.exp((couple[1] - max(values)) * beta) / Z))

    return maxxed

def positional_embedder_(x: str):
    x = list(x)
    string = ""

    for i in range(len(x)):
        string += f"{x[i]}.{i} "
    string += '§ '


    return string

class UnidimensionalAggregator:
    def __init__(self):
        self.dict = dict()
        self.X = []

    def fit(self, X, separator=''):
        """X is a lot of words"""
        self.X = X
        for word in self.X:
            split = positional_embedder_(word).split(" ")
            split1 = split[:-1]
            split2 = split[1:]
            for b, a in zip(split1, split2):
                if b in self.dict:
                    if a in self.dict[b]:
                        self.dict[b][a] += 1
                    else:
                        self.dict[b][a] = 1
                else:
                    self.dict[b] = {a: 1}

    def predict(self, X):
        """X is a token"""

        softmaxxed = softmaxdict(self.dict[X], beta=0.01)
        predictions = rd.choices([s[0] for s in softmaxxed], weights=[s[1] for s in softmaxxed], k=1)
        return predictions

    def autoregressive_predict(self, X, steps=10):
        """X is a token"""
        predictions = [X]
        for b in range(steps):
            key = predictions[-1]
            if "§" in key:
                predictions = predictions[:-1]
                break

            pred = self.predict(key)[0]
            predictions.append(pred)

        return predictions[1:]

agg = UnidimensionalAggregator()
datas = []
with open("human_names.txt") as f:
    for line in f.readlines():
        datas.append(line.strip())

agg.fit(datas)
start = 'b.0'
pred = agg.autoregressive_predict(start, 100)
preds = [p.split('.')[0] for p in pred]
name_ = start.split('.')[0] + "".join(preds)
print(name_)
print(name_ in datas) # this checks if the model does not output names already in the dataset

brila
False


# Obsevations
The cool thing here is that the model trains itself every run, its incredibly fast and funny
We also want to underline the fact that the models predict when to end the name.

# Is deep learning more suitable for this problem?
We now try to use a CBOW style model trained on tokens from a very simple tokenizer, and then from a BPE.
To solve this problem we used the same Positional Embedding.



In [15]:
import torch  # let's do this here, to break the wall of text
torch.manual_seed(42)

### Training data

Download the file *names.txt* from the course GitHub page.

From the sidebar on the left, click on the folder icon at the very bottom.

Drag the *human_names.txt* and *pokemon_names.txt* files into the folder and wait for the upload to complete.

In [16]:
names = open('human_names.txt', 'r').read().splitlines()

len(names)  # should be 32033 for human names, 1302 for Pokémon names

32033

### Name encoding and decoding

Your trained model must be able to digest and process text characters. You'll do this by writing a simple encoder/decoder such that:

```"hello" -> [13, 2, 5, 5, 7]```

...and of course, the opposite direction.

The specific numbers are not important, but you need your `encode` and `decode` functions to behave correctly:

```decode(encode(s)) == s```

for any string `s`.

Here the simple Tokenizer model, we hope we can code the bpe from scratch today as well

In [100]:
# Supponiamo che utf8_dict sia definito così (per test semplici):
utf8_dict = {chr(i): chr(i) for i in range(97, 123)}  # solo lettere minuscole a-z


In [101]:
class Tokenizer:
  def __init__(self, text, threshold = 700):
    # Creazione del dizionario dei caratteri UTF-8
    self.freq = {v:1 for v in utf8_dict.values()}
    self.text = text
    self.tokenized_text = []
    self.threshold = threshold
    # Let's suppose ds contains list of things like: testi = ['mario', 'alessio', 'antonio', 'luca', 'alessandro', 'emanuele', 'gabriele', 'marco']
    self.batches = []
    self.bs = 60
    self.n_batches = len(self.text)//self.bs
    print('Threshold:\n', self.threshold)
    print('Number of batches:\n', self.n_batches)

    for i in range(self.n_batches):
      self.batches.append(self.text[i*self.bs : (i+1)*self.bs])

  def count(self):
    lista_tokens = [t for tokens in self.tokenized_text for t in tokens]
    self.freq = self.freq | {v : self.freq[v] + 1 for v in lista_tokens}

  def tokenize_str(self, string):
      tok_text.append([])

      end = len(word)
      end_t = end
      start = 0

      while start < end:
        while word[start : end_t] not in tokens and end_t > start:
           end_t -= 1
        if end_t == start:
           end_t = start + 1


        tokenized_text[i].append(word[start : end_t])
        start = end_t
        end_t = end
        self.tokenized_text = tokenized_text

    return tokenized_text

  def set_to_one(self):
    tokens = self.freq.keys()
    for k in tokens:
      self.freq[k] = 1

  def tokenize(self, text):
    tokens = self.freq.keys()
    list_text = text
    tokenized_text = []

    for i, word  in enumerate(list_text):
      tokenized_text.append([])

      end = len(word)
      end_t = end
      start = 0

      while start < end:
        while word[start : end_t] not in tokens and end_t > start:
           end_t -= 1
        if end_t == start:
           end_t = start + 1


        tokenized_text[i].append(word[start : end_t])
        start = end_t
        end_t = end
        self.tokenized_text = tokenized_text

    return tokenized_text


  def add_token(self, text):
    list_text = text
    tokenized_text = self.tokenize(text)
    self.count()
    freq_pairs = {}
    score = {}

    for word in tokenized_text:

      num_tok = len(word)

      for i in range(num_tok-1):
        pair = (word[i], word[i + 1])
        if pair not in freq_pairs:
          freq_pairs[pair] = 1
        else:
          freq_pairs[pair] += 1

        score[pair] = freq_pairs[pair]/(self.freq[pair[0]] * self.freq[pair[1]])

    pair_max = max(score, key = lambda x : score[x])
    new_token = pair_max[0] + pair_max[1]
    self.freq[new_token] = 1

    return score

  def to_tok(self, text):
    for i in range(self.threshold//self.n_batches):
      score = self.add_token(text)
      self.set_to_one()


  def to_numb(self):
    tokens = self.freq.keys()
    to_num = {k : i for i, k in enumerate(tokens)}

    return to_num

  def main_to_tok(self):
    for batch in self.batches:
      self.to_tok(batch)
    print("Updated frequencies:\n", tok.freq)


In [102]:
names = open('human_names.txt', 'r').read().splitlines()

In [103]:
# Inizializza il tokenizer
tok = Tokenizer(names)

print('Final run',tok.main_to_tok())
print('ID tokens:\n', tok.to_numb())
print('Number of tokens:\n', len(tok.to_numb().keys()))


Threshold:
 700
Number of batches:
 533
Updated frequencies:
 {'a': 1, 'b': 1, 'c': 1, 'd': 1, 'e': 1, 'f': 1, 'g': 1, 'h': 1, 'i': 1, 'j': 1, 'k': 1, 'l': 1, 'm': 1, 'n': 1, 'o': 1, 'p': 1, 'q': 1, 'r': 1, 's': 1, 't': 1, 'u': 1, 'v': 1, 'w': 1, 'x': 1, 'y': 1, 'z': 1, 'el': 1, 'na': 1, 'le': 1, 'la': 1, 'li': 1, 'ri': 1, 'an': 1, 'ar': 1, 'ma': 1, 'or': 1, 'yn': 1, 'lee': 1, 'lan': 1, 'al': 1, 'ka': 1, 'ne': 1, 'en': 1, 'ya': 1, 'ra': 1, 'in': 1, 'ad': 1, 'lei': 1, 'am': 1, 'ie': 1, 'lyn': 1, 'ia': 1, 'on': 1, 'er': 1, 'tt': 1, 'ana': 1, 'os': 1, 're': 1, 'mar': 1, 'da': 1, 'as': 1, 'is': 1, 'st': 1, 'ja': 1, 'ley': 1, 'ry': 1, 'iya': 1, 'iyah': 1, 'ina': 1, 'bry': 1, 'ni': 1, 'ch': 1, 'ee': 1, 'leig': 1, 'yah': 1, 'ah': 1, 'il': 1, 'es': 1, 'kay': 1, 'av': 1, 'ha': 1, 'mi': 1, 'ig': 1, 'sh': 1, 'ca': 1, 'yla': 1, 'ke': 1, 'th': 1, 'za': 1, 'rie': 1, 'bri': 1, 'den': 1, 'ay': 1, 'lie': 1, 'ey': 1, 'em': 1, 'jo': 1, 'lo': 1, 'ko': 1, 'lu': 1, 'nna': 1, 'ira': 1, 'ss': 1, 'lynn': 1, 'k

### Dataset and data loaders

The training data should simply be a bunch of pairs `(char_in, char_out)`, since you want `M(char_in) = char_out`.

In [91]:

tok.main_to_tok()

contexts = []
for nametok in namestok:
  for i in range(len(nametok) - 2):
      contexts.append(torch.tensor((nametok[i], nametok[i + 1], nametok[i+2])))


Updated frequencies:
 {'a': 1, 'b': 1, 'c': 1, 'd': 1, 'e': 1, 'f': 1, 'g': 1, 'h': 1, 'i': 1, 'j': 1, 'k': 1, 'l': 1, 'm': 1, 'n': 1, 'o': 1, 'p': 1, 'q': 1, 'r': 1, 's': 1, 't': 1, 'u': 1, 'v': 1, 'w': 1, 'x': 1, 'y': 1, 'z': 1, 'el': 1, 'na': 1, 'le': 1, 'la': 1, 'li': 1, 'ri': 1, 'an': 1, 'ar': 1, 'ma': 1, 'or': 1, 'yn': 1, 'lee': 1, 'lan': 1, 'al': 1, 'ka': 1, 'ne': 1, 'en': 1, 'ya': 1, 'ra': 1, 'in': 1, 'ad': 1, 'lei': 1, 'am': 1, 'ie': 1, 'lyn': 1, 'ia': 1, 'on': 1, 'er': 1, 'tt': 1, 'ana': 1, 'os': 1, 're': 1, 'mar': 1, 'da': 1, 'as': 1, 'is': 1, 'st': 1, 'ja': 1, 'ley': 1, 'ry': 1, 'iya': 1, 'iyah': 1, 'ina': 1, 'bry': 1, 'ni': 1, 'ch': 1, 'ee': 1, 'leig': 1, 'yah': 1, 'ah': 1, 'il': 1, 'es': 1, 'kay': 1, 'av': 1, 'ha': 1, 'mi': 1, 'ig': 1, 'sh': 1, 'ca': 1, 'yla': 1, 'ke': 1, 'th': 1, 'za': 1, 'rie': 1, 'bri': 1, 'den': 1, 'ay': 1, 'lie': 1, 'ey': 1, 'em': 1, 'jo': 1, 'lo': 1, 'ko': 1, 'lu': 1, 'nna': 1, 'ira': 1, 'ss': 1, 'lynn': 1, 'ki': 1, 'anna': 1, 'nn': 1, 'sa': 1, 'ys'

Create your `NGramDataset` class and `DataLoader`s for training, validation and test:

In [92]:
dataloader = torch.utils.data.DataLoader(contexts, batch_size=64, shuffle=True)

In [93]:
print(next(iter(dataloader)))

tensor([[ 67,  74, 152],
        [  0,   1,  88],
        [ 36,  46,  47],
        [ 18,  42,   5],
        [138,  58, 169],
        [207,  21,   5],
        [ 14,  37,  43],
        [ 18,  58,   5],
        [ 39,  96,  72],
        [ 12,  35,  92],
        [ 39,  69,   3],
        [ 39,  40,  28],
        [  7,   8,  75],
        [ 38,  23,  64],
        [122,  68, 123],
        [ 39, 145,   3],
        [ 11, 109,   5],
        [ 44,  66,  21],
        [ 67,  35,  53],
        [ 47,  48,   5],
        [  3,  83,  89],
        [ 19,  51,  77],
        [  6,   7,  36],
        [ 20, 114, 115],
        [ 39,   2,  55],
        [  9,  18,  44],
        [112,  47,   5],
        [ 75,  95,   5],
        [ 55, 135,   5],
        [ 30,  66,  21],
        [140, 104, 113],
        [ 19,  63, 176],
        [ 58,  66,   5],
        [ 85,  10, 211],
        [  3, 188,   5],
        [  1,  36,  28],
        [ 37,  18,  89],
        [ 43,  44,  66],
        [ 69, 139,  47],
        [ 56,  36,  61],


### Model

Now create your training model!

The model we choose to use is a CBOW style model that operates on our positional aware tokens. Though we tried this dangerous idea of using a variation embedding.
So we trained two different embeddings, one for the mean, and one for the variance as in a $\beta$VAE.

$$z = \sum \mu + \sum \sigma \varepsilon$$

where

$$\varepsilon \sim N(0, I)$$

Then the loss we choose to be a L1 norm (to induce sparsity) + KL divergence with $N(0, I)$


In [104]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class CBOW_VAE(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim

        self.embedding_mean = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_log_var = nn.Embedding(vocab_size, embedding_dim)
        self.decoder1 = nn.Linear(embedding_dim, embedding_dim*4)
        self.decoder2 = nn.Linear(embedding_dim*4, vocab_size)

        self.act1 = nn.LeakyReLU(0.2)
        self.act2 = nn.LeakyReLU(0.2)
        self.act4 = nn.Sigmoid()

    def forward(self, x):
        mean = self.embedding_mean(x)
        log_var = self.embedding_log_var(x)

        mean = mean.sum(dim=1)
        log_var = log_var.sum(dim=1)

        z = mean + torch.randn_like(log_var) * torch.exp(log_var)

        z = self.decoder1(z)
        z = self.act1(z)
        z = self.decoder2(z)
        z = self.act4(z)

        return z, mean, log_var

model = CBOW_VAE(len(tok.to_numb()), 128)


### Training

Time to train!

In [ ]:
def KLLL1Loss(y_pred, y_true, mean, logvar, beta=0.1):
    MSE = torch.nn.functional.l1_loss(y_pred, y_true, reduction='mean')

    KL = -0.5 * torch.sum(1 + logvar - mean.pow(2) - torch.exp(logvar), dim=1)

    loss = (MSE + beta * KL).mean()
    return loss

In [105]:
epochs = 1
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(epochs):
  for i, batch in enumerate(dataloader):
    x = torch.tensor(batch[:, :-1])
    y = torch.tensor(batch[:, -1:])
    optimizer.zero_grad()
    y_pred, mean, logvar = model(x)
    # loss = loss_fn(y_pred, torch.nn.functional.one_hot(y, len(tok.dict)), mean, logvar)
    loss = loss_fn(y_pred.float(), torch.nn.functional.one_hot(y, len(tok.to_numb())).float())
    loss.backward()
    optimizer.step()
    if i % 1000 == 0:
      print(f"Epoch {epoch}, batch {i}, loss {loss.item()}")

<ipython-input-105-cbdd78a4807f>:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(batch[:, :-1])
<ipython-input-105-cbdd78a4807f>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(batch[:, -1:])
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64, 1, 559])) that is different to the input size (torch.Size([64, 559])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 0, batch 0, loss 0.3104422688484192
Epoch 0, batch 1000, loss 0.0017258377047255635
Epoch 0, batch 2000, loss 0.0016691513592377305


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([48, 1, 559])) that is different to the input size (torch.Size([48, 559])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


### Inference

Test your model and generate new crazy names!

In [84]:
input = "dar"
input = tok.tokenize(positional_embedder(input, False))[:-1]
print(input)
word = input

for i in range(10):
  _in = [input[-2:]]
  out, _, __ = model(torch.tensor(_in))
  print(torch.argmax(out))
  break
  input.append(torch.argmax(out))

print(tok.detokenize(input))

out = tok.detokenize(input)
print(out)
out = "".join([o.split('.')[0] for o in out.split()])
print(out)

[125, 39, 40]
tensor(5)
d.0 a.1 r.2
d.0 a.1 r.2
dar


In [106]:
inp = 'dar'
inptok = Tokenizer([inp])
print(inp)

Threshold:
 700
Number of batches:
 0
dar


Probably we experimented a little too much with the model, but the output with this little time does not seem to beat the simple Positional Aware Gibbs Sampling which we called PAGS (very original)
We could totally could work more on the CBOW VAE and other models (also simpler models), but we had no time to do so :(


### Larger context

Now that you have a working basic model, generalize it so that it can take as input *more than one character*; this should improve the quality of the generated names!

For example, using a context length of `3`:

`M(".an") = "t"`

...may eventually get you `"anthony"`, because the inference steps see a longer context that can better condition the generation.

Instead, our current context length of `1`:

`M(".") = "a"`

...may diverge and generate unrealistic names like `axyzyll`.

### Pokémon names

We managed to generate these artificial Pokémon names:
- beakwily
- mordar
- dortytel
- bymanlona
- amlozinder

Can you do better than us? 😇

Download the *pokemon_names.txt* file from the course webpage and try to come up with an appropriate solution to beat our names!

Go catch them all!

In [ ]:
# here